In [3]:
import torch
import torch.nn as nn
import numpy as np

#B -> batch ,  E -> input dimension, H -> hidden  dimension,  T -> series stpe
B, E, H = 1, 128, 3

In [4]:
def prepare_input():
    np.random.seed(42)
    vocab = {"播放": 0, "周杰伦": 1, "的": 2, "《稻香》": 3}
    tokens = ["播放", "周杰伦", "的", "《稻香》"]
    ids = [vocab[t] for t in tokens]

    # word vector(V, E)
    V = len(vocab)
    em_table = np.random.randn(V, E).astype(np.float32)
    # print(em_table[0][:2])
    # print(em_table[1][:2])

    # x_np.shape -> (1, 4, 128), here [None]=[np.newaxis]
    x_np = em_table[ids][None]
    """
    x_np = em_table[ids][np.newaxis]
    x_np = em_table[ids].reshape[1, 4, 128]
    x_np = np.expand_dims(em_table[ids], axis=0)
    """
    # print(em_table[1].shape)
    return tokens, x_np

In [5]:
def manual_run_numpy(x_np, U_np, W_np ):
    """
    h_t = tanh(U x_t + w h_{t-1})
    x_np: {B, T, E)
    u_np: (E, H)
    W_np: (H, H)
    return:
    output: (B, T, H)
    final_h: (B, H)
    """
    B_local, T_local, _ = x_np.shape
    h_prev = np.zeros((B_local, H), dtype=np.float32)
    steps = []
    for t in range(T_local):
        # -> [1, 128]
        x_t = x_np[:, t, :]
        # (1,128) * (128, 3) = (1, 3), (1,3) * (3, 3) = (1, 3)
        h_t = np.tanh(x_t @ U_np + h_prev @ W_np)
        steps.append(h_t)
        h_prev = h_t
    # (1,3) -> (1, 4,3)
    outputs = np.stack(steps, axis=1)
    return outputs, h_prev



In [6]:
def pytorch_run_forward(x, U, W):
    rnn = nn.RNN(
        input_size=E,
        hidden_size=H,
        num_layers=1,
        nonlinearity='tanh',
        bias=False,
        batch_first=True,
        bidirectional=False
    )
    # inject the customized weight to rnn , l0 layer0
    with torch.no_grad():
        rnn.weight_ih_l0.copy_(U.T)
        rnn.weight_hh_l0.copy_(W.T)
    y, h_n = rnn(x)
    return y, h_n.squeeze(0)

In [7]:
_, x_np = prepare_input()
x = torch.from_numpy(x_np).float()
torch.manual_seed(7)
U = torch.randn(E, H)
W = torch.randn(H, H)
print(U.shape)
print(W.shape)


torch.Size([128, 3])
torch.Size([3, 3])


In [8]:
U_np = U.detach().numpy()
W_np = W.detach().numpy()
print(type(U_np), U_np.shape)

<class 'numpy.ndarray'> (128, 3)


In [9]:
out_manual_up, ht_manual_up = manual_run_numpy(x_np, U_np, W_np)
print(out_manual_up.shape, ht_manual_up.shape)

(1, 4, 3) (1, 3)


In [10]:
out_torch, ht_torch = pytorch_run_forward(x, U, W)
print(out_torch.shape, ht_torch.shape)

torch.Size([1, 4, 3]) torch.Size([1, 3])


In [14]:
out_manual = torch.from_numpy(out_manual_up)
ht_manual = torch.from_numpy(ht_manual_up)

print(torch.allclose(out_torch, out_manual,atol=1e-4))
print(torch.allclose(ht_manual, ht_torch, atol=1e-1))


True
True


In [15]:
print(ht_manual)
print(ht_torch)

tensor([[ 0.9999, -1.0000, -0.9966]])
tensor([[ 0.9999, -1.0000, -0.9966]], grad_fn=<SqueezeBackward1>)


In [16]:
print(out_torch)
print(out_manual)

tensor([[[ 0.0607,  1.0000,  0.9857],
         [-1.0000, -1.0000,  0.7301],
         [-1.0000, -1.0000, -1.0000],
         [ 0.9999, -1.0000, -0.9966]]], grad_fn=<TransposeBackward1>)
tensor([[[ 0.0607,  1.0000,  0.9857],
         [-1.0000, -1.0000,  0.7301],
         [-1.0000, -1.0000, -1.0000],
         [ 0.9999, -1.0000, -0.9966]]])


In [17]:
print(out_torch[:, -1, :])
print(ht_torch)

tensor([[ 0.9999, -1.0000, -0.9966]], grad_fn=<SelectBackward0>)
tensor([[ 0.9999, -1.0000, -0.9966]], grad_fn=<SqueezeBackward1>)
